# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR2):

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

> All individual dataset entities (record sets, fields, columns, etc.) are referenced using their `@id`, ensuring unambiguous cross-referencing per the FAIR2 Croissant schema.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records with `mlcroissant`. The Croissant file fully describes the schema and available data tables.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
# Access the metadata directly as an object
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's list available `RecordSet` entities, their `@id`, and their fields. These define the logical tables in the dataset, each with a set of fields (columns).

In [ ]:
# List details of all record sets and their fields by @id
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"Name: {rs.name}")
    print(f"Description: {getattr(rs, 'description', '')}")
    print("Fields (by @id):")
    for field in rs.fields:
        print(f" - {field.id} ({field.name}) - Type: {getattr(field, 'data_type', 'unknown')}")

## 3. Data Extraction
Let's load data from one or more record sets into pandas DataFrames for inspection and analysis. Here, all record sets will be loaded and stored in a dictionary keyed by their `@id`.

> You can refer to individual tables via their `@id` keys below.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

# Show available record set IDs and display the first few records from the first table
if len(record_set_ids) == 0:
    print("No record sets found in the metadata.")
else:
    print("Loaded record sets (by @id):")
    for i, rs_id in enumerate(record_set_ids):
        print(f"{i+1}. {rs_id}")
    main_record_set_id = record_set_ids[0]
    df_main = dataframes[main_record_set_id]
    print(f"\nColumns in '{main_record_set_id}':\n{df_main.columns.tolist()}")
    display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic analyses: filter by a numeric field, normalize it, and group by a categorical attribute.

Below, update the `numeric_field_id` and `group_field_id` variables to match the `@id` of a numeric and a grouping field as printed in the previous section for your chosen record set.

> For example, use `peptide_count` or `coverage_percent` as a numeric field and `sample_type` or other categorical columns as a grouping field depending on your record set.

In [ ]:
# Choose the main record set ID (update if needed)
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Set these to the appropriate @id for a numeric and a grouping field (replace example values)
numeric_field_id = 'coverage_percent'  # Example only, use actual @id value from the overview above
group_field_id = 'sample_type'         # Example only

if numeric_field_id in df.columns:
    threshold = 10  # Filter threshold, adjust as appropriate
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (count={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped mean values by '{group_field_id}':")
        display(grouped_df)
    else:
        print(f"Grouping field '{group_field_id}' not found in columns.")
else:
    print(f"Numeric field '{numeric_field_id}' not found. Please set the variable to match a column @id from your record set.")

## 5. Visualization
Let's visualize the distribution of the numeric field (histogram) and its relation to the grouping field (box plot), if present.

> If you're running in a notebook environment, the following visualizations will display inline.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We've loaded the FAIR<sup>2</sup> dataset, explored its structure via `mlcroissant`, and performed basic EDA and visualizations. Remember to adjust field IDs for your analysis to your schema's actual field `@id` as necessary.

**Summary**: This workflow demonstrates loading data directly from a FAIR-compliant Croissant schema; reviewing table and column structure; and performing initial data inspection, processing, and visualization—all while referencing schema elements by their `@id`.